In [1]:
import pandas as pd
import numpy as np

In [2]:
# Ingest granular master dataset to generate optimized visualization layer structs
df = pd.read_parquet('../01_data/04_engineered/shipment_features.parquet')
print(f"Ingested master dataset with {len(df):,} rows. Commencing pre-aggregation")

Ingested master dataset with 10,194 rows. Commencing pre-aggregation


In [3]:
# DATASET 1: SPATIAL HEATMAP OPTIMIZATION

# Pre-aggregate spatial performance indicators to reduce UI rendering overhead
dash_state_heatmap = df.groupby(['state_province', 'region']).agg(
    total_sales=('sales', 'sum'),
    total_units=('units', 'sum'),
    avg_lead_time=('shipping_lead_time', 'mean'),
    delay_rate=('is_delayed', 'mean')
).reset_index()

In [4]:
# Enforce uniform rounding limits to support clean tooltip rendering
dash_state_heatmap['total_sales'] = dash_state_heatmap['total_sales'].round(2)
dash_state_heatmap['avg_lead_time'] = dash_state_heatmap['avg_lead_time'].round(2)
dash_state_heatmap['delay_rate'] = (dash_state_heatmap['avg_lead_time'] * 100).round(2)

In [5]:
# DATASET 2: ROUTE EFFICIENCY LEADERBOARD

# Generate hierarchy elements required for interactive drill-throughs
dash_route_leaderboard = df.groupby(['route_state', 'factory', 'state_province']).agg(
    total_shipments=('order_id', 'count'),
    avg_lead_time=('shipping_lead_time', 'mean'),
    std_lead_time=('shipping_lead_time', 'std'),
    delayed_shipments=('is_delayed', 'sum'),
    total_sales=('sales', 'sum')
).reset_index()

In [6]:
dash_route_leaderboard['route_efficiency_score'] = (
    (dash_route_leaderboard['total_shipments'] - dash_route_leaderboard['delayed_shipments'])
    / dash_route_leaderboard['total_shipments'] * 100
).round(2)

In [7]:
dash_route_leaderboard['avg_lead_time'] = dash_route_leaderboard['avg_lead_time'].round(2)
dash_route_leaderboard['total_sales'] = dash_route_leaderboard['total_sales'].round(2)

In [8]:
# Handle invalid standard deviations resulting from single-shipment route populations
dash_route_leaderboard['std_lead_time'] = dash_route_leaderboard['std_lead_time'].fillna(0).round(2)

In [9]:
# Drop intermediary variables to enforce minimal payload sizes
dash_route_leaderboard = dash_route_leaderboard.drop(columns=['delayed_shipments'])

In [10]:
# DATASET 3: CARRIER TIER PERFORMANCE

# Compile dimensional metrics assessing service-level compliance globally
dash_ship_mode = df.groupby(['ship_mode', 'region', 'factory']).agg(
    total_orders=('order_id', 'count'),
    avg_lead_time=('shipping_lead_time', 'mean'),
    delay_rate=('is_delayed', 'mean')
).reset_index()

In [11]:
dash_ship_mode['avg_lead_time'] = dash_ship_mode['avg_lead_time'].round(2)
dash_ship_mode['delay_rate'] = (dash_ship_mode['delay_rate'] * 100).round(2)

In [12]:
# GATE VALIDATIONS & EXPORT
print("\n--- PERFORMANCE VALIDATION REPORT ---")
print(f"Heatmap Dataset Size: {len(dash_state_heatmap)} rows (Compressed from {len(df)})")
print(f"Leaderboard Dataset Size: {len(dash_route_leaderboard)} rows")
print(f"Ship Mode Dataset Size: {len(dash_ship_mode)} rows")


--- PERFORMANCE VALIDATION REPORT ---
Heatmap Dataset Size: 59 rows (Compressed from 10194)
Leaderboard Dataset Size: 196 rows
Ship Mode Dataset Size: 74 rows


In [13]:
# Reconcile shipment volumetric counts to prevent silent data loss during aggregation
assert dash_route_leaderboard['total_shipments'].sum() == len(df), "Data Loss Warning: Order totals do not match master records!"
print("Validation Passed: Aggregated volumes exactly match row-level operational counts.")

Validation Passed: Aggregated volumes exactly match row-level operational counts.


In [14]:
# Export formalized Gold-tier outputs ready for framework usage (Streamlit/Dash/etc)
dash_state_heatmap.to_parquet('../01_data/05_dashboard/dash_state_heatmap.parquet', index=False)
dash_route_leaderboard.to_parquet('../01_data/05_dashboard/dash_route_leaderboard.parquet', index=False)
dash_ship_mode.to_parquet('../01_data/05_dashboard/dash_ship_mode.parquet', index=False)

print("\nSUCCESS: Gold Layer exported. Dashboard data structures are fully optimized.")


SUCCESS: Gold Layer exported. Dashboard data structures are fully optimized.
